# Train Models

Trains 5 detectors on the **train** split, tunes the main hyper-parameter on the **validation** split (never on test) and saves everything to `outputs/models/`.

Needs: notebook 1 finished. **Next:** `3_Test_Evaluation.ipynb`

In [ ]:
OUT_DIR = "outputs"
SEED    = 42

In [ ]:
import os, re, glob, time, json, warnings, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
import joblib
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)

OUT = Path(OUT_DIR)
for sub in ["figures", "tables", "models", "data", "generated"]: (OUT / sub).mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.25, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})

def savefig(name):
    plt.savefig(OUT / "figures" / f"{name}.png"); plt.savefig(OUT / "figures" / f"{name}.pdf"); plt.show()

def save_table(df, name, index=True):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    try: (OUT / "tables" / f"{name}.tex").write_text(df.to_latex(index=index, float_format="%.4f"))
    except Exception: pass
    return df
print("Setup OK")

In [ ]:
# ---- shared helpers (fake = positive class; label 1 = real, 0 = fake) ----
def trunc(s, n): return " ".join(str(s).split()[:n])

def load_splits():
    f = OUT / "data" / "splits.pkl"
    if not f.exists(): raise FileNotFoundError("Run notebook 1 (Data Preparation) first.")
    return joblib.load(f)

_pat = json.loads((OUT / "data" / "scrub_patterns.json").read_text()) if (OUT / "data" / "scrub_patterns.json").exists() else []
LEAK_RE = re.compile("|".join(_pat), flags=re.I) if _pat else None
def scrub(s): return re.sub(r"\s+", " ", LEAK_RE.sub(" ", s)).strip() if LEAK_RE else s

def fake_scores(model, X):
    """higher = more likely FAKE."""
    if hasattr(model, "predict_proba"): return model.predict_proba(X)[:, 0]
    return -model.decision_function(X)

def fake_pred(model, X):
    return (fake_scores(model, X) > (0.5 if hasattr(model, "predict_proba") else 0)).astype(int)

def metrics(y, model, X):
    yf = (y == 0).astype(int); s = fake_scores(model, X); p = fake_pred(model, X)
    return dict(Accuracy=accuracy_score(yf, p), Precision=precision_score(yf, p), Recall=recall_score(yf, p),
                F1=f1_score(yf, p), AUC=roc_auc_score(yf, s))

def stylometric(df, title_col="title", text_col="text"):
    t, x = df[text_col].astype(str), df[title_col].astype(str)
    nw = t.str.split().str.len().clip(lower=1); nc = t.str.len().clip(lower=1)
    f = pd.DataFrame(index=df.index)
    f["n_words"] = nw; f["n_chars"] = nc; f["avg_word_len"] = nc / nw
    f["n_sent"] = t.str.count(r"[.!?]+").clip(lower=1); f["avg_sent_len"] = nw / f["n_sent"]
    f["upper_ratio"] = t.str.count(r"[A-Z]") / nc; f["caps_word_ratio"] = t.str.count(r"\b[A-Z]{3,}\b") / nw
    f["exclam"] = t.str.count("!") / nw; f["question"] = t.str.count(r"\?") / nw
    f["quote"] = t.str.count(r"[\"“”']") / nw; f["digit_ratio"] = t.str.count(r"\d") / nc
    f["punct_ratio"] = t.str.count(r"[^\w\s]") / nc
    f["ttr"] = t.map(lambda s: len(set(s.lower().split())) / max(1, len(s.split())))
    f["title_words"] = x.str.split().str.len(); f["title_upper"] = x.str.count(r"[A-Z]") / x.str.len().clip(lower=1)
    f["title_excl"] = x.str.count("!")
    return f.astype(np.float32)

In [ ]:
splits = load_splits(); train, val = splits["train"], splits["val"]
ytr, yva = train.label.values, val.label.values
print(f"train={len(train):,}  val={len(val):,}")

### 2.1 Feature extraction (TF-IDF word 1–2 grams + 15 stylometric features)

In [ ]:
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, max_features=150_000, sublinear_tf=True, dtype=np.float32)
Xtr = tfidf.fit_transform(train.doc); Xva = tfidf.transform(val.doc)
Str, Sva = stylometric(train), stylometric(val)
print(f"TF-IDF matrix {Xtr.shape} built in {time.time()-t0:.0f}s (this is the slowest cell, ~1-2 min)")

### 2.2 Hyper-parameter tuning on the validation set

In [ ]:
grid = {"LogReg (TF-IDF)":     (lambda c: LogisticRegression(C=c, solver="liblinear", random_state=SEED), [1, 10, 100]),
        "Linear SVM (TF-IDF)": (lambda c: LinearSVC(C=c, random_state=SEED), [0.1, 0.5, 2]),
        "SGD-Huber (TF-IDF)":  (lambda a: SGDClassifier(loss="modified_huber", alpha=a, max_iter=30, random_state=SEED), [1e-5, 2e-5, 1e-4]),
        "Compl. NB (TF-IDF)":  (lambda a: ComplementNB(alpha=a), [0.1, 0.3, 1.0])}
tune, best_param = [], {}
for name, (mk, vals) in grid.items():
    for v in vals:
        m = mk(v).fit(Xtr, ytr); tune.append({"Model": name, "Param": v, "Val F1": metrics(yva, m, Xva)["F1"]})
tune = pd.DataFrame(tune); best_param = tune.loc[tune.groupby("Model")["Val F1"].idxmax()].set_index("Model")["Param"].to_dict()
save_table(tune.round(4), "t3a_tuning_validation", index=False); display(tune.round(4)); print("Chosen:", best_param)

### 2.3 Final training + validation results

In [ ]:
models = {n: (grid[n][0](best_param[n]), "tfidf") for n in grid}
models["HistGB (Stylometric)"] = (HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, random_state=SEED), "style")
X = {"tfidf": (Xtr, Xva), "style": (Str, Sva)}
fitted, rows = {}, []
for name, (m, kind) in models.items():
    t0 = time.time(); m.fit(X[kind][0], ytr); tt = time.time() - t0
    rows.append({"Model": name, **metrics(yva, m, X[kind][1]), "Train s": round(tt, 1)}); fitted[name] = (m, kind)
    print(f"{name:24s} trained in {tt:5.1f}s")
res_val = pd.DataFrame(rows).set_index("Model"); save_table(res_val.round(4), "t3b_validation_results"); display(res_val.round(4))

In [ ]:
res_val[["Accuracy", "Precision", "Recall", "F1", "AUC"]].plot(kind="bar", figsize=(10, 4.5), rot=20, width=.8)
plt.ylim(max(0.5, float(res_val[["Accuracy", "Precision", "Recall", "F1", "AUC"]].values.min()) - 0.03), 1.005)
plt.title("Validation performance of the 5 detectors (fake = positive)"); plt.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.3))
plt.tight_layout(); savefig("fig2a_validation_comparison")

### 2.4 Save trained models

In [ ]:
joblib.dump({"tfidf": tfidf, "models": {k: v[0] for k, v in fitted.items()}, "kinds": {k: v[1] for k, v in fitted.items()}, "params": best_param},
            OUT / "models" / "phase2_models.joblib")
print("Saved ->", OUT / "models" / "phase2_models.joblib", "\n✅ Notebook 2 finished. Now open 3_Test_Evaluation.ipynb")